# 업무추진비 데이터 시각화

In [25]:
# import os
# import random #데이터 샘플링
# from collections import Counter # count 용도

import numpy as np
import pandas as pd

from tqdm import tqdm

# 결측치 확인
import missingno as msno

import warnings
warnings.filterwarnings('ignore')

# 시각화
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
plt.style.use('fivethirtyeight')

# 한글, 마이너스 깨짐 방지
from matplotlib import rc, font_manager, rcParams
font=font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
rc('font', family = font)
rcParams['axes.unicode_minus'] = False

# 지도
# from geopy import distance # 거리 계산
# import geopy.distance
import folium
from folium.plugins import HeatMap

# plotly
import ipywidgets as widgets
from ipywidgets import interact

# 이걸 설정하면 Multiple Output이 가능함
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import chart_studio.plotly as py 
import plotly.express as px
import cufflinks as cf 
cf.go_offline(connected=True)

## 데이터 불러오기

In [9]:
df = pd.read_csv('./dataset/업무추진비_지도.csv')
df

,사용장소,방문횟수,사용금액,카테고리,주소,lat,lng
0,가마솥,4,224000,보양식,서울 관악구 관악로17길 13 가마솥,37.480721,126.951002
1,감나무집,33,6527500,한식,서울 관악구 관악로11길 20 감나무집,37.477906,126.951002
2,강강술래,21,5068500,고기,서울 관악구 남부순환로 1660 강강술래,37.484604,126.935058
3,강촌민물매운탕2호점,7,547000,해물,서울 동작구 보라매로5길 15 강촌민물매운탕2호점,37.498098,126.920755
4,갯바위,6,458000,해물,서울 관악구 남부순환로216길 13 갯바위,37.480840,126.949110
...,...,...,...,...,...,...,...
114,해물나라,17,1892000,해물,서울 관악구 남부순환로 1674 해물나라,37.469685,126.933227
115,홍콩반점,7,278500,중식,서울 관악구 관악로 152 홍콩반점,37.478742,126.952557
116,황복,6,390000,보양식,서울 관악구 봉천로 356 황복,37.483903,126.939363
117,후포리,4,535000,해물,서울 관악구 봉천로 388 후포리,37.482873,126.943077


## folium으로 시각화해보기

In [12]:
# 관악구청을 중심으로 객체 생성하기
m = folium.Map([37.478408041522954, 126.9514186693129], zoom_start=10)
m

In [13]:
# 지도에 마커로 음식점 표시하기
for i in df.index:
    sub_lat =  df.loc[i, 'lat']
    sub_lng = df.loc[i, 'lng']
    
    title = df.loc[i, '사용장소']
    
    folium.Marker([sub_lat, sub_lng], tooltip=title).add_to(m)

m.save('sample.html')
m

In [20]:
# 그린파크식당 확인
df[df['사용장소']=='그린파크식당']

,사용장소,방문횟수,사용금액,카테고리,주소,lat,lng
9,그린파크식당,5,1600000,한식,인천 옹진군 백령면 백령로297번길 16 그린파크식당,37.971058,124.715432


+ 그린파크 식당은 제외해주자

In [23]:
# 그린파크식당 제거
df.drop(9, axis=0, inplace=True)
df.index = range(len(df))
df

,사용장소,방문횟수,사용금액,카테고리,주소,lat,lng
0,가마솥,4,224000,보양식,서울 관악구 관악로17길 13 가마솥,37.480721,126.951002
1,감나무집,33,6527500,한식,서울 관악구 관악로11길 20 감나무집,37.477906,126.951002
2,강강술래,21,5068500,고기,서울 관악구 남부순환로 1660 강강술래,37.484604,126.935058
3,강촌민물매운탕2호점,7,547000,해물,서울 동작구 보라매로5길 15 강촌민물매운탕2호점,37.498098,126.920755
4,갯바위,6,458000,해물,서울 관악구 남부순환로216길 13 갯바위,37.480840,126.949110
...,...,...,...,...,...,...,...
113,해물나라,17,1892000,해물,서울 관악구 남부순환로 1674 해물나라,37.469685,126.933227
114,홍콩반점,7,278500,중식,서울 관악구 관악로 152 홍콩반점,37.478742,126.952557
115,황복,6,390000,보양식,서울 관악구 봉천로 356 황복,37.483903,126.939363
116,후포리,4,535000,해물,서울 관악구 봉천로 388 후포리,37.482873,126.943077


### 히트맵 그려보기

In [24]:
m = folium.Map(
    location = [36.5053542, 127.7043419],
    zoom_start = 8,
    tiles = 'Cartodb Positron'
#     tiles='stamentoner'
)

data = []

for i in df.index:
    
    sub_lat =  df.loc[i, 'lat']
    sub_lng = df.loc[i, 'lng']
    
    data.append([sub_lat, sub_lng])

# 관악구청 마커로 표시
folium.Marker([37.478408041522954, 126.9514186693129], tooltip='관악구청').add_to(m)
    

    
HeatMap(data).add_to(m)

m.save('heat.html')
m

In [26]:
def visualize_by_date(df):
    interact(aca_map, date=list(np.sort(df['date'].unique())))

In [ ]:
list(np.sort(df['date'].unique())

In [30]:
def map_by_tax(df):
    def view_map(cat):
        data = df
        m = aca_map(zoom=15)
        for i in data.index:
            lat = data.loc[i, 'lat']
            lng = data.loc[i, 'lng']
            name = data.loc[i, '사용장소']
            folium.Marker([lat, lng], icon=folium.Icon(
                color='orange', icon='heart-empty'), popup=name).add_to(m)
        return m
    interact(view_map, 
#              ZMS = widgets.IntSlider(min=0, max=100, step=1, value=30), 
             cat = list(df.카테고리.unique()))
#              dis = widgets.IntSlider(min=0, max=max(df.distance), step=10, value=700))
    


In [31]:
map_by_tax(df)

interactive(children=(Dropdown(description='cat', options=('보양식', '한식', '고기', '해물', '중식', '국밥류', '분식', '카페&디저트…